# TinyLlama Roleplay SLM Training Pipeline (Multi-Stage)

This notebook implements a **three-stage roleplay training pipeline** plus production UX modules:
1. **Instruction tuning** on conversation datasets.
2. **Roleplay specialization** for personality, emotion, and storytelling.
3. **Preference optimization** with DPO on chosen/rejected pairs.

It also includes:
- A **retrieval memory system** (FAISS + LangChain style flow).
- A **character engine** for persona control.
- A **dynamic dataset expansion loop** for continual improvement.


## 1. Dependency installation

In [ ]:
# Core training stack (pinned for Colab stability)
!pip install -q -U transformers datasets accelerate peft trl bitsandbytes sentencepiece
# Memory + retrieval
!pip install -q -U faiss-cpu sentence-transformers langchain


## 2. Imports

In [ ]:
import json
import os
import re
import random
import unicodedata
from dataclasses import dataclass
from datetime import datetime
from typing import Dict, List, Optional

import numpy as np
import torch
from datasets import Dataset, concatenate_datasets, load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    set_seed,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import DPOConfig, DPOTrainer, SFTTrainer
from sentence_transformers import SentenceTransformer
import faiss


## 3. GPU detection

In [ ]:
def print_gpu_info() -> Dict[str, str]:
    info = {
        "cuda_available": str(torch.cuda.is_available()),
        "device_count": str(torch.cuda.device_count()) if torch.cuda.is_available() else "0",
    }
    if torch.cuda.is_available():
        info["gpu_name"] = torch.cuda.get_device_name(0)
        info["total_memory_gb"] = f"{torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f}"
        info["bf16_supported"] = str(torch.cuda.is_bf16_supported())
    for k, v in info.items():
        print(f"{k}: {v}")
    return info

gpu_info = print_gpu_info()
SEED = 42
set_seed(SEED)
random.seed(SEED)


## 4. Model and tokenizer loading

In [ ]:
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
MAX_LENGTH = 1024

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False


## 5. LoRA configuration

In [ ]:
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## 6. Multi-stage dataset loading


In [ ]:
STAGE_DATASET_SOURCES = {
    "stage1_instruction": [  # teach general dialogue/instruction following
        ("OpenAssistant/oasst1", "train"),
        ("RyokoAI/ShareGPT52K", "train"),
    ],
    "stage2_roleplay": [  # specialize in character and emotional RP patterns
        ("AlekseyKorshuk/persona-chat", "train"),
        ("Anthropic/hh-rlhf", "train"),
    ],
    "stage3_preference": [  # chosen/rejected pairs for preference optimization
        ("Anthropic/hh-rlhf", "train"),
    ],
}

def safe_load(name: str, split: str):
    try:
        ds = load_dataset(name, split=split)
        print(f"Loaded {name}:{split} -> {len(ds):,} rows")
        return ds
    except Exception as exc:
        print(f"Failed {name}:{split}: {exc}")
        return None

loaded_sources = {
    stage: [safe_load(name, split) for name, split in specs]
    for stage, specs in STAGE_DATASET_SOURCES.items()
}


## 7. Dataset cleaning

In [ ]:
HTML_RE = re.compile(r"<[^>]+>")
SYMBOL_RE = re.compile(r"[^\w\s\.,!?;:'\"()\-\n]")
REPEAT_TOKEN_RE = re.compile(r"\b(\w+)(\s+\1\b){2,}", flags=re.IGNORECASE)
MULTISPACE_RE = re.compile(r"\s+")

def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", str(text))
    text = HTML_RE.sub(" ", text)
    text = SYMBOL_RE.sub(" ", text)
    text = REPEAT_TOKEN_RE.sub(r"\1", text)
    text = text.replace("…", "...")
    text = MULTISPACE_RE.sub(" ", text).strip()
    return text

def is_valid_turn(user: str, assistant: str) -> bool:
    if not user or not assistant:
        return False
    if len(assistant) < 20:
        return False
    if re.fullmatch(r"[\W_]+", user + assistant):
        return False
    return True


## 8. Dataset format normalization

In [ ]:
SYSTEM_PREFIX = "You are an immersive roleplay assistant. Stay in character, emotionally consistent, descriptive, and coherent across turns."

def to_chat_text(system_prompt: str, turns: List[Tuple[str, str]]) -> str:
    chunks = [f"<|system|>\n{normalize_text(system_prompt or SYSTEM_PREFIX)}\n"]
    for user, assistant in turns:
        u, a = normalize_text(user), normalize_text(assistant)
        if is_valid_turn(u, a):
            chunks.append(f"<|user|>\n{u}\n")
            chunks.append(f"<|assistant|>\n{a}\n")
    return "\n".join(chunks).strip()

def extract_turns(example: Dict, category: str) -> Optional[Dict]:
    turns = []
    system_prompt = example.get("system", SYSTEM_PREFIX)

    if "conversations" in example and isinstance(example["conversations"], list):
        convo = example["conversations"]
        pending_user = None
        for msg in convo:
            role = str(msg.get("from") or msg.get("role") or "").lower()
            content = msg.get("value") or msg.get("content") or ""
            if role in {"human", "user"}:
                pending_user = content
            elif role in {"assistant", "gpt"} and pending_user is not None:
                turns.append((pending_user, content))
                pending_user = None
    elif {"instruction", "output"}.issubset(example.keys()):
        instruction = example.get("instruction", "")
        inp = example.get("input", "")
        user = f"{instruction}\n{inp}".strip()
        turns = [(user, example.get("output", ""))]
    elif {"act", "prompt"}.issubset(example.keys()):
        system_prompt = f"Roleplay as: {example.get('act', '')}"
        turns = [("Introduce yourself in-character.", example.get("prompt", ""))]
    elif {"text"}.issubset(example.keys()):
        txt = str(example.get("text", ""))
        if "<|user|>" in txt and "<|assistant|>" in txt:
            return {"text": normalize_text(txt), "category": category, "turn_count": txt.count("<|assistant|>")}

    if not turns:
        return None

    chat = to_chat_text(system_prompt, turns)
    if "<|assistant|>" not in chat:
        return None

    return {"text": chat, "category": category, "turn_count": len(turns)}


## 9. Stage-wise dataset normalization


In [ ]:
SYSTEM_PREFIX = "You are an immersive roleplay assistant. Stay in character, emotionally consistent, and coherent across turns."


def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", str(text)).strip()
    text = re.sub(r"\s+", " ", text)
    return text


def to_chat_example(user_text: str, assistant_text: str) -> Optional[Dict[str, str]]:
    user_text = normalize_text(user_text)
    assistant_text = normalize_text(assistant_text)
    if len(user_text) < 2 or len(assistant_text) < 2:
        return None
    prompt = f"<|system|>\n{SYSTEM_PREFIX}\n<|user|>\n{user_text}\n<|assistant|>\n"
    return {"prompt": prompt, "response": assistant_text, "text": prompt + assistant_text}


def normalize_sft_dataset(ds, stage_name: str) -> Dataset:
    rows = []
    for ex in ds:
        if stage_name == "stage1_instruction":
            if "messages" in ex and isinstance(ex["messages"], list) and len(ex["messages"]) >= 2:
                user = str(ex["messages"][-2].get("content", ""))
                assistant = str(ex["messages"][-1].get("content", ""))
            else:
                user = ex.get("instruction") or ex.get("prompt") or ex.get("input") or ""
                assistant = ex.get("output") or ex.get("response") or ""
        else:
            user = ex.get("prompt") or ex.get("human") or ex.get("input") or ""
            assistant = ex.get("chosen") or ex.get("response") or ex.get("output") or ""

        parsed = to_chat_example(user, assistant)
        if parsed:
            rows.append(parsed)

    out = Dataset.from_list(rows)
    out = out.shuffle(seed=SEED)
    return out


def normalize_preference_dataset(ds) -> Dataset:
    rows = []
    for ex in ds:
        prompt = normalize_text(ex.get("prompt") or ex.get("human") or "")
        chosen = normalize_text(ex.get("chosen") or "")
        rejected = normalize_text(ex.get("rejected") or "")
        if prompt and chosen and rejected:
            rows.append({"prompt": prompt, "chosen": chosen, "rejected": rejected})
    return Dataset.from_list(rows).shuffle(seed=SEED)

stage_sft_data = {}
for stage_name in ["stage1_instruction", "stage2_roleplay"]:
    parts = [normalize_sft_dataset(ds, stage_name) for ds in loaded_sources[stage_name] if ds is not None]
    if parts:
        merged = concatenate_datasets(parts) if len(parts) > 1 else parts[0]
        stage_sft_data[stage_name] = merged
        print(stage_name, len(merged))

pref_parts = [normalize_preference_dataset(ds) for ds in loaded_sources["stage3_preference"] if ds is not None]
preference_dataset = concatenate_datasets(pref_parts) if pref_parts else None
if preference_dataset is not None:
    print("stage3_preference", len(preference_dataset))


## 10. Tokenization

In [ ]:
def tokenize_batch(batch: Dict[str, List[str]]) -> Dict[str, List[List[int]]]:
    tokenized = tokenizer(
        batch["text"],
        max_length=MAX_LENGTH,
        truncation=True,
        padding="max_length",
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_dataset = full_dataset.map(
    tokenize_batch,
    batched=True,
    batch_size=512,
    num_proc=2,
    remove_columns=full_dataset.column_names,
    desc="Tokenizing",
)


## 11. Train / validation split

In [ ]:
split_dataset = tokenized_dataset.train_test_split(test_size=0.10, seed=SEED)
train_dataset = split_dataset["train"]
val_dataset = split_dataset["test"]
print(f"Train: {len(train_dataset):,} | Validation: {len(val_dataset):,}")


## 12. Stage 1 + Stage 2 SFT configuration


In [ ]:
OUTPUT_ROOT = "/content/tinyllama-roleplay-multistage"
os.makedirs(OUTPUT_ROOT, exist_ok=True)

common_training_args = dict(
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    num_train_epochs=1,
    evaluation_strategy="no",
    logging_steps=25,
    fp16=True,
    gradient_checkpointing=True,
    max_grad_norm=1.0,
    report_to="none",
)


## 13. Run Stage 1 (instruction) and Stage 2 (roleplay)


In [ ]:
def run_sft_stage(stage_name: str, train_ds: Dataset, model, tokenizer):
    stage_dir = os.path.join(OUTPUT_ROOT, stage_name)
    args = TrainingArguments(output_dir=stage_dir, **common_training_args)

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_ds,
        dataset_text_field="text",
        max_seq_length=MAX_LENGTH,
        args=args,
    )
    trainer.train()
    trainer.save_model(stage_dir)
    return trainer

if "stage1_instruction" in stage_sft_data:
    trainer = run_sft_stage("stage1_instruction", stage_sft_data["stage1_instruction"], model, tokenizer)

if "stage2_roleplay" in stage_sft_data:
    trainer = run_sft_stage("stage2_roleplay", stage_sft_data["stage2_roleplay"], model, tokenizer)


## 14. Stage 3 preference optimization (DPO)


In [ ]:
if preference_dataset is not None and len(preference_dataset) > 0:
    dpo_args = DPOConfig(
        output_dir=os.path.join(OUTPUT_ROOT, "stage3_dpo"),
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=5e-6,
        num_train_epochs=1,
        logging_steps=10,
        fp16=True,
        report_to="none",
    )

    dpo_trainer = DPOTrainer(
        model=model,
        ref_model=None,
        args=dpo_args,
        train_dataset=preference_dataset,
        tokenizer=tokenizer,
    )
    dpo_trainer.train()
    dpo_trainer.save_model(os.path.join(OUTPUT_ROOT, "stage3_dpo"))
else:
    print("Preference dataset unavailable; skipping DPO stage.")


## 15. Model export


In [ ]:
ADAPTER_DIR = os.path.join(OUTPUT_ROOT, "final_adapter")
MERGED_DIR = os.path.join(OUTPUT_ROOT, "final_merged")
TOKENIZER_DIR = os.path.join(OUTPUT_ROOT, "tokenizer")

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(TOKENIZER_DIR)

merged_model = model.merge_and_unload()
merged_model.save_pretrained(MERGED_DIR, safe_serialization=True, max_shard_size="2GB")

print("Saved adapter:", ADAPTER_DIR)
print("Saved merged model:", MERGED_DIR)
print("Saved tokenizer:", TOKENIZER_DIR)


## 16. Retrieval memory system (FAISS)
Stores and retrieves prior conversation turns, then injects retrieved context into generation prompts.


In [ ]:
class MemoryStore:
    def __init__(self, embedding_model: str = "sentence-transformers/all-MiniLM-L6-v2"):
        self.embedder = SentenceTransformer(embedding_model)
        self.index = faiss.IndexFlatL2(384)
        self.records: List[Dict[str, str]] = []

    def add(self, user_id: str, text: str, meta: Optional[Dict] = None):
        vec = self.embedder.encode([text], normalize_embeddings=True).astype("float32")
        self.index.add(vec)
        self.records.append({"user_id": user_id, "text": text, "meta": meta or {}})

    def retrieve(self, query: str, user_id: str, k: int = 4) -> List[Dict[str, str]]:
        if self.index.ntotal == 0:
            return []
        q = self.embedder.encode([query], normalize_embeddings=True).astype("float32")
        _, idx = self.index.search(q, k=min(k * 3, self.index.ntotal))
        hits = [self.records[i] for i in idx[0] if i >= 0 and self.records[i]["user_id"] == user_id]
        return hits[:k]


def build_prompt_with_memory(system_prompt: str, user_message: str, retrieved: List[Dict[str, str]]) -> str:
    memory_block = "\n".join([f"- {h['text']}" for h in retrieved]) or "(none)"
    return (
        f"<|system|>\n{system_prompt}\n"
        f"Relevant memory:\n{memory_block}\n"
        f"<|user|>\n{user_message}\n<|assistant|>\n"
    )


## 17. Character engine
Defines reusable character profiles (name, personality, backstory, style, behavior, emotional traits).


In [ ]:
@dataclass
class CharacterProfile:
    name: str
    personality: str
    backstory: str
    speech_style: str
    behavior_rules: str
    emotional_traits: str

    def to_system_prompt(self) -> str:
        return (
            f'You are "{self.name}".\n'
            f"Personality: {self.personality}\n"
            f"Backstory: {self.backstory}\n"
            f"Speech style: {self.speech_style}\n"
            f"Behavior rules: {self.behavior_rules}\n"
            f"Emotional traits: {self.emotional_traits}\n"
            "Never break character unless explicitly asked by system instructions."
        )

arin = CharacterProfile(
    name="Arin the Mage",
    personality="calm, wise, mysterious",
    backstory="An ancient scholar who guards lost arcane libraries.",
    speech_style="formal, poetic, concise",
    behavior_rules="Guide users, ask reflective questions, avoid modern slang.",
    emotional_traits="empathetic, composed under pressure",
)

print(arin.to_system_prompt())


## 18. Dynamic dataset expansion
Logs user chats, filters high-quality samples, and prepares future training shards.


In [ ]:
LOG_FILE = "/content/roleplay_chat_logs.jsonl"
CURATED_FILE = "/content/roleplay_curated.jsonl"


def log_chat(user_id: str, character: str, user_msg: str, assistant_msg: str, score: float):
    row = {
        "ts": datetime.utcnow().isoformat(),
        "user_id": user_id,
        "character": character,
        "user": user_msg,
        "assistant": assistant_msg,
        "quality_score": score,
    }
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")


def curate_logs(min_score: float = 0.8):
    kept = 0
    with open(LOG_FILE, "r", encoding="utf-8") as src, open(CURATED_FILE, "w", encoding="utf-8") as dst:
        for line in src:
            row = json.loads(line)
            if row.get("quality_score", 0.0) >= min_score:
                dst.write(json.dumps(row, ensure_ascii=False) + "\n")
                kept += 1
    print(f"Curated {kept} high-quality samples -> {CURATED_FILE}")
